In [1]:
import os
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import re

In [2]:
num_workers = 4  
fastq_dir = Path("./fastq")  
symlink_dir = Path("./symlinks") # Temporary clean-named links
#results_dir = Path("results")
snp_only_dir = Path("./snp_only")
logs_dir = Path("./logs")
tmp_dir = Path("./tmp")
vcf_dir = Path("./vcf")

In [3]:
#results_dir.mkdir(exist_ok=True)
symlink_dir.mkdir(exist_ok=True)
snp_only_dir.mkdir(exist_ok=True)
logs_dir.mkdir(exist_ok=True)
tmp_dir.mkdir(exist_ok=True)

In [ ]:
# Group FASTQs by ENA run ID
samples = {}

for fq in fastq_dir.glob("*.fastq.gz"):
    name = fq.name

    # Match ERR / SRR with regex
    match = re.search(r'([SE]RR\d+)', name)
    if not match:
        print(f"⚠️ Skipping {name}: no ENA run ID found.")
        continue

    clean_base = match.group(1)

    # Figure out _1 or _2
    if "_1" in name:
        suffix = "_1"
    elif "_2" in name:
        suffix = "_2"
    else:
        print(f"⚠️ Skipping {name}: no read pair info.")
        continue

    # Make symlink with correct name
    symlink_name = f"{clean_base}{suffix}.fastq.gz"
    symlink_path = symlink_dir / symlink_name

    if not symlink_path.exists():
        symlink_path.symlink_to(fq.resolve())

    samples.setdefault(clean_base, []).append(symlink_path)

print(f"✅ Found {len(samples)} samples to process.")

⚠️ Skipping Bangladesh_SAMN08436049_SRR6650271.fastq.gz: no read pair info.
⚠️ Skipping Bangladesh_SAMN08436163_SRR6650422.fastq.gz: no read pair info.
⚠️ Skipping Bangladesh_SAMN08436240_SRR6650224.fastq.gz: no read pair info.
⚠️ Skipping Bangladesh_SAMN08629079_SRR6797692.fastq.gz: no read pair info.
⚠️ Skipping Bangladesh_SAMN08629369_SRR6797638.fastq.gz: no read pair info.
⚠️ Skipping Canada_site.24.subj.PT-15.lab.2007-412.iso.1.fastq.gz: no ENA run ID found.
⚠️ Skipping Canada_site.24.subj.PT-347.lab.2013-021.iso.1.fastq.gz: no ENA run ID found.
⚠️ Skipping Canada_site.24.subj.PT-47.lab.1999-442.iso.1.fastq.gz: no ENA run ID found.
⚠️ Skipping Canada_site.24.subj.PT-711.lab.2019-160.iso.1.fastq.gz: no ENA run ID found.
⚠️ Skipping China_SAMN03653238_SRR2024933.fastq.gz: no read pair info.
⚠️ Skipping China_SAMN03653268_SRR2024966.fastq.gz: no read pair info.
⚠️ Skipping China_SAMN03653298_SRR2024999.fastq.gz: no read pair info.
⚠️ Skipping Germany_site.03.subj.10247-02_LIB9207.lab

In [ ]:
# Function to process one sample
def process_sample(sample, files):
    files = sorted(files)
    output_prefix = sample

    if len(files) == 2:
        fq1, fq2 = files
        tb_cmd = [
            "tb-profiler",
            "profile",
            "-1", str(fq1),
            "-2", str(fq2),
            "-p", str(output_prefix),
            "--txt",
            "--temp", "./tmp"
        ]
    elif len(files) == 1:
        fq1 = files[0]
        tb_cmd = [
            "tb-profiler",
            "profile",
            "-1", str(fq1),
            "-p", str(output_prefix),
            "--txt",
            "--temp", "./tmp"
        ]
    else:
        print(f"⚠️ Skipping {sample}: no valid FASTQ files.")
        return

    print(f"🔬 Running TB-Profiler for {sample}")
    with open(logs_dir / f"{sample}_tbprofiler.log", "w") as log_file:
        subprocess.run(tb_cmd, stdout=log_file, stderr=log_file, check=True)

    input_vcf = vcf_dir / f"{output_prefix}.targets.vcf.gz"
    output_vcf = snp_only_dir / f"{sample}_snps_only.vcf"

    bcf_cmd = [
        "bcftools", "view",
        "-v", "snps",
        input_vcf,
        "-o", str(output_vcf),
        "--output-type", "v"
    ]

    print(f"🧬 Extracting SNPs for {sample}")
    with open(logs_dir / f"{sample}_bcftools.log", "w") as log_file:
        subprocess.run(bcf_cmd, stdout=log_file, stderr=log_file, check=True)

    print(f"✅ Finished: {sample}")

In [ ]:
# Run with parallel jobs 
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = []
    for sample, files in samples.items():
        futures.append(executor.submit(process_sample, sample, files))

    for future in as_completed(futures):
        try:
            future.result()
        except subprocess.CalledProcessError as e:
            print(f"❌ Error running command: {e}") # meron pa ring error like ERR4810720, ERR2199934, SRR6152742 figure out why

print("🎉 All samples done.")

🔬 Running TB-Profiler for ERR4830728
🔬 Running TB-Profiler for SRR6650271
🔬 Running TB-Profiler for SRR6650422
🔬 Running TB-Profiler for SRR6650224
🧬 Extracting SNPs for SRR6650224
✅ Finished: SRR6650224
🔬 Running TB-Profiler for SRR6797692
🧬 Extracting SNPs for SRR6650271
✅ Finished: SRR6650271
🔬 Running TB-Profiler for SRR6797638
🧬 Extracting SNPs for SRR6650422
✅ Finished: SRR6650422
🔬 Running TB-Profiler for ERR5917676
🧬 Extracting SNPs for ERR4830728
✅ Finished: ERR4830728
🔬 Running TB-Profiler for ERR5917689
🧬 Extracting SNPs for ERR5917676
✅ Finished: ERR5917676
🔬 Running TB-Profiler for ERR2516329
🧬 Extracting SNPs for SRR6797638
✅ Finished: SRR6797638
🔬 Running TB-Profiler for ERR2516383
🧬 Extracting SNPs for SRR6797692
✅ Finished: SRR6797692
🔬 Running TB-Profiler for SRR6153242
🧬 Extracting SNPs for ERR2516329
✅ Finished: ERR2516329
🔬 Running TB-Profiler for SRR6153217
🧬 Extracting SNPs for ERR5917689
✅ Finished: ERR5917689
🔬 Running TB-Profiler for SRR6152742
🧬 Extracting SN